<a href="https://www.kaggle.com/code/ugokorubeast/ugoko-byu?scriptVersionId=315947157" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 目次

- [1. 環境設定と入力探索](#sec-setup)
- [2. tomo 抽出](#sec-select)
- [3. 前処理（raw Image -> .npy）](#sec-preprocess)
- [4. CustomDataset と座標→ラベルマップ変換](#sec-dataset)
- [5. モデル構築](#sec-build-model)
  - [5-1. BasicBlock](#sec-build-basicblock)
  - [5-2. Bottleneck](#sec-build-bottleneck)
  - [5-3. Encoder](#sec-build-encoder)
  - [5-4. Decoder Common Block](#sec-build-decoder-common)
  - [5-5. Decoder](#sec-build-decoder)
  - [5-6. Net Constructor I/O](#sec-build-net-constructor)
- [6. 学習・評価ループ（2-5）](#sec-train)

この目次から各章へ移動できます。

## 1. 環境設定と入力探索
<a id="sec-setup"></a>

ライブラリ読み込み、`BaseConfig`（2-4）で学習設定を一元化、seed 固定、入力ディレクトリ探索を行います。

In [15]:
from pathlib import Path
from dataclasses import dataclass
import random
import hashlib

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ===== 2-4: Base configuration class =====
@dataclass
class BaseConfig:
    seed: int = 42
    num_tomos: int = 4
    depth: int = 16
    input_size: int = 96
    batch_size: int = 1
    num_workers: int = 0
    epochs: int = 3
    lr: float = 1e-3
    label_radius: int = 1


CFG = BaseConfig()

# Backward-compatible aliases for existing cells
SEED = CFG.seed
NUM_TOMOS = CFG.num_tomos
DEPTH = CFG.depth
IMG_SIZE = CFG.input_size
BATCH_SIZE = CFG.batch_size
NUM_WORKERS = CFG.num_workers
EPOCHS = CFG.epochs
LR = CFG.lr
LABEL_RADIUS = CFG.label_radius

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ../input, ./input, train_data などから tomo_* ディレクトリがある場所を自動検出
candidate_roots = [
    Path('../input'),
    Path('../input/competitions/byu-locating-bacterial-flagellar-motors-2025/train'),
    Path('./input'),
    Path('train_data'),
    Path('../train_data'),
]

input_root = None
for root in candidate_roots:
    if root.exists() and any(p.is_dir() and p.name.startswith('tomo_') for p in root.iterdir()):
        input_root = root
        break

if input_root is None:
    raise FileNotFoundError(
        "tomo_* ディレクトリが見つかりません。候補: ../input, ./input, train_data, ../train_data"
    )

print(f"[INFO] input_root = {input_root.resolve()}")
print(f"[INFO] CFG = {CFG}")

[INFO] input_root = /Users/yuhaogao/workspace/BYU-competition/train_data
[INFO] CFG = BaseConfig(seed=42, num_tomos=4, depth=16, input_size=96, batch_size=1, num_workers=0, epochs=3, lr=0.001, label_radius=1)


## 2. tomo 抽出
<a id="sec-select"></a>

利用可能な tomo から seed 固定で学習対象を抽出します。

In [16]:
all_tomos = sorted([
    p for p in input_root.iterdir()
    if p.is_dir() and p.name.startswith('tomo_')
])

if len(all_tomos) < NUM_TOMOS:
    raise ValueError(f"tomo フォルダ数が不足しています: found={len(all_tomos)}, required={NUM_TOMOS}")

# seed 固定で 4 件ランダム抽出
selected_tomos = random.sample(all_tomos, k=NUM_TOMOS)

print('[INFO] selected_tomos:')
for p in selected_tomos:
    print(' -', p.name)

[INFO] selected_tomos:
 - tomo_2c607f
 - tomo_2bb588
 - tomo_4925ee
 - tomo_f6a38a


## 3. 前処理（raw Image -> .npy）
<a id="sec-preprocess"></a>

生データのスライス画像を読み込み、学習で使いやすい `.npy` ボリュームとして保存します。

In [17]:
processed_root = Path('./processed_tomos')
processed_root.mkdir(parents=True, exist_ok=True)


def load_and_resize_volume(tomo_dir: Path, img_size: int):
    slice_paths = sorted(tomo_dir.glob('slice_*.jpg'))
    if not slice_paths:
        slice_paths = sorted(tomo_dir.glob('*.jpg'))
    if not slice_paths:
        raise FileNotFoundError(f'画像スライスが見つかりません: {tomo_dir}')

    volume = []
    for path in slice_paths:
        image = Image.open(path).convert('L').resize((img_size, img_size))
        array = np.asarray(image, dtype=np.float32) / 255.0
        volume.append(array)

    return np.stack(volume, axis=0)  # [D, H, W]


# 2-1: 生データ -> .npy 前処理
# NOTE: まずは選択した tomo のみを保存。全件処理するなら preprocess_targets = all_tomos に変更。
preprocess_targets = selected_tomos

for tomo_dir in preprocess_targets:
    volume = load_and_resize_volume(tomo_dir, IMG_SIZE)
    out_path = processed_root / f'{tomo_dir.name}.npy'
    np.save(out_path, volume.astype(np.float32))
    print(f'[PREPROCESS] {tomo_dir.name} -> {out_path} shape={volume.shape}')

print(f'[INFO] processed_root = {processed_root.resolve()}')

[PREPROCESS] tomo_2c607f -> processed_tomos/tomo_2c607f.npy shape=(300, 96, 96)
[PREPROCESS] tomo_2bb588 -> processed_tomos/tomo_2bb588.npy shape=(300, 96, 96)
[PREPROCESS] tomo_4925ee -> processed_tomos/tomo_4925ee.npy shape=(300, 96, 96)
[PREPROCESS] tomo_f6a38a -> processed_tomos/tomo_f6a38a.npy shape=(500, 96, 96)
[INFO] processed_root = /Users/yuhaogao/workspace/BYU-competition/processed_tomos


## 4. CustomDataset と座標→ラベルマップ変換
<a id="sec-dataset"></a>

CSV 座標を読み取り、3D ラベルマップへ変換して input/target を返す Dataset を作成し、`get_dataset` / `get_dataloader` で DataLoader を組み立てます。

In [18]:
class CustomDataset(Dataset):
    """2-2 実装: 座標アノテーションを 3D ラベルマップへ変換して返す。"""

    def __init__(self, tomo_dirs, depth=16, img_size=96, label_radius=1, processed_root: Path | None = None):
        self.tomo_dirs = list(tomo_dirs)
        self.depth = depth
        self.img_size = img_size
        self.label_radius = label_radius
        self.processed_root = processed_root
        self.coord_meta = self._load_coord_metadata()

    def __len__(self):
        return len(self.tomo_dirs)

    def _load_slice(self, path: Path):
        img = Image.open(path).convert('L').resize((self.img_size, self.img_size))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return arr

    def _load_coord_metadata(self):
        csv_candidates = [
            Path('./folds_all.csv'),
            Path('./data/processed/folds_all.csv'),
            Path('../input/competitions/byu-locating-bacterial-flagellar-motors-2025/train_labels.csv'),
        ]
        csv_path = next((p for p in csv_candidates if p.exists()), None)
        if csv_path is None:
            print('[WARN] folds_all.csv が見つからないため、ラベルは全て 0 になります。')
            return {}

        df = pd.read_csv(csv_path)
        grouped = {}
        for tomo_id, g in df.groupby('tomo_id'):
            first = g.iloc[0]
            coords = []
            # 列名に空白があるため、属性アクセスではなく列参照で座標を取得する
            for z, y, x in g[['Motor axis 0', 'Motor axis 1', 'Motor axis 2']].to_numpy(dtype=float):
                if z >= 0 and y >= 0 and x >= 0:
                    coords.append((z, y, x))

            grouped[tomo_id] = {
                'orig_shape': (
                    int(first['Array shape (axis 0)']),
                    int(first['Array shape (axis 1)']),
                    int(first['Array shape (axis 2)']),
                ),
                'coords': coords,
            }
        return grouped

    def _scale_coord(self, coord, src_len, dst_len):
        if src_len <= 1:
            return 0
        return int(round(coord * (dst_len - 1) / (src_len - 1)))

    def _load_volume_from_images(self, tomo_dir: Path):
        slices = sorted(tomo_dir.glob('slice_*.jpg'))
        if not slices:
            slices = sorted(tomo_dir.glob('*.jpg'))
        if not slices:
            raise FileNotFoundError(f'画像スライスが見つかりません: {tomo_dir}')

        # z 方向は全体から等間隔サンプリングして depth に揃える
        if len(slices) >= self.depth:
            sample_idx = np.linspace(0, len(slices) - 1, self.depth).astype(int)
            selected = [slices[i] for i in sample_idx]
        else:
            selected = slices + [slices[-1]] * (self.depth - len(slices))

        return np.stack([self._load_slice(p) for p in selected], axis=0)  # [D, H, W]

    def _match_depth(self, vol):
        d = vol.shape[0]
        if d == self.depth:
            return vol
        if d > self.depth:
            idx = np.linspace(0, d - 1, self.depth).astype(int)
            return vol[idx]
        pad = np.repeat(vol[-1:, :, :], self.depth - d, axis=0)
        return np.concatenate([vol, pad], axis=0)

    def _load_volume(self, tomo_dir: Path):
        # 2-1 の出力を優先して読み込み、ない場合のみ raw 画像から作る
        if self.processed_root is not None:
            npy_path = self.processed_root / f'{tomo_dir.name}.npy'
            if npy_path.exists():
                vol = np.load(npy_path).astype(np.float32)
                return self._match_depth(vol)

        return self._load_volume_from_images(tomo_dir)

    def _make_label_map(self, tomo_id: str):
        label = np.zeros((self.depth, self.img_size, self.img_size), dtype=np.float32)
        meta = self.coord_meta.get(tomo_id)
        if meta is None or len(meta['coords']) == 0:
            return label

        src_d, src_h, src_w = meta['orig_shape']
        r = self.label_radius
        for z0, y0, x0 in meta['coords']:
            z = self._scale_coord(z0, src_d, self.depth)
            y = self._scale_coord(y0, src_h, self.img_size)
            x = self._scale_coord(x0, src_w, self.img_size)

            z_min, z_max = max(0, z - r), min(self.depth, z + r + 1)
            y_min, y_max = max(0, y - r), min(self.img_size, y + r + 1)
            x_min, x_max = max(0, x - r), min(self.img_size, x + r + 1)
            label[z_min:z_max, y_min:y_max, x_min:x_max] = 1.0

        return label

    def __getitem__(self, idx):
        tomo_dir = self.tomo_dirs[idx]
        vol = self._load_volume(tomo_dir)
        label = self._make_label_map(tomo_dir.name)

        x = torch.from_numpy(vol).unsqueeze(0)      # [1, D, H, W]
        target = torch.from_numpy(label).unsqueeze(0)  # [1, D, H, W]
        return {'input': x, 'target': target, 'tomo_id': tomo_dir.name}


# 2-3: get_dataset / get_dataloader 実装
# mode ごとの既定値を分けつつ、shuffle/batch/num_workers を明示的に扱う
DATALOADER_CFG = {
    'train': {
        'batch_size': BATCH_SIZE,
        'shuffle': True,
        'num_workers': NUM_WORKERS,
    },
    'val': {
        'batch_size': BATCH_SIZE,
        'shuffle': False,
        'num_workers': NUM_WORKERS,
    },
}


def get_dataset(tomo_dirs, mode='train'):
    return CustomDataset(
        tomo_dirs,
        depth=DEPTH,
        img_size=IMG_SIZE,
        label_radius=LABEL_RADIUS,
        processed_root=processed_root,
    )


def get_dataloader(dataset, mode='train', shuffle=None, batch_size=None, num_workers=None):
    cfg = DATALOADER_CFG.get(mode, DATALOADER_CFG['train'])

    if shuffle is None:
        shuffle = cfg['shuffle']
    if batch_size is None:
        batch_size = cfg['batch_size']
    if num_workers is None:
        num_workers = cfg['num_workers']

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
    )


# 2-5: train/valid 比較のための最小 split（固定順）
if len(selected_tomos) >= 2:
    train_tomos = selected_tomos[:-1]
    val_tomos = selected_tomos[-1:]
else:
    train_tomos = selected_tomos
    val_tomos = selected_tomos

train_ds = get_dataset(train_tomos, mode='train')
val_ds = get_dataset(val_tomos, mode='val')

train_loader = get_dataloader(train_ds, mode='train')
val_loader = get_dataloader(val_ds, mode='val')

sample = next(iter(train_loader))
print('[INFO] train_loader cfg =', DATALOADER_CFG['train'])
print('[INFO] val_loader cfg =', DATALOADER_CFG['val'])
print('[INFO] train_tomos =', [p.name for p in train_tomos])
print('[INFO] val_tomos =', [p.name for p in val_tomos])
print('[INFO] sample input shape =', tuple(sample['input'].shape))
print('[INFO] sample target shape =', tuple(sample['target'].shape))
print('[INFO] sample target positive voxels =', int(sample['target'].sum().item()))

[INFO] train_loader cfg = {'batch_size': 1, 'shuffle': True, 'num_workers': 0}
[INFO] val_loader cfg = {'batch_size': 1, 'shuffle': False, 'num_workers': 0}
[INFO] train_tomos = ['tomo_2c607f', 'tomo_2bb588', 'tomo_4925ee']
[INFO] val_tomos = ['tomo_f6a38a']
[INFO] sample input shape = (1, 1, 16, 96, 96)
[INFO] sample target shape = (1, 1, 16, 96, 96)
[INFO] sample target positive voxels = 27


## 5. モデル構築
<a id="sec-build-model"></a>

### 5-1. BasicBlock
<a id="sec-build-basicblock"></a>

BasicBlock の実装と downsample 分岐の検証を行います。

In [19]:
# ===== 3-1-3: BasicBlock downsample branch implementation =====
from timm.layers import DropPath


def conv3x3x3(ic, oc, stride=1):
    return nn.Conv3d(
        in_channels=ic,
        out_channels=oc,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False,
    )


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=False, drop_path_rate=0.0):
        super().__init__()
        self.conv1 = conv3x3x3(inplanes, planes, stride=stride)
        self.bn1 = nn.BatchNorm3d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3x3(planes, planes, stride=1)
        self.bn2 = nn.BatchNorm3d(planes)
        self.drop_path = DropPath(drop_prob=drop_path_rate) if drop_path_rate > 0.0 else nn.Identity()

        if isinstance(downsample, nn.Module):
            self.downsample = downsample
        elif downsample:
            self.downsample = nn.Sequential(
                nn.Conv3d(
                    inplanes,
                    planes * self.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm3d(planes * self.expansion),
            )
        else:
            self.downsample = None

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.drop_path(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out = out + residual
        out = self.relu(out)
        return out


# 3-1-3 validation: downsample=True with stride/channel change
x_ds = torch.randn(2, 16, 8, 32, 32)
block_ds = BasicBlock(inplanes=16, planes=32, stride=2, downsample=True, drop_path_rate=0.1)
block_ds.eval()
y_ds = block_ds(x_ds)
print(f"[downsample] input shape:  {tuple(x_ds.shape)}")
print(f"[downsample] output shape: {tuple(y_ds.shape)}")
assert y_ds.shape == (2, 32, 4, 16, 16)

[downsample] input shape:  (2, 16, 8, 32, 32)
[downsample] output shape: (2, 32, 4, 16, 16)


### 5-2. Bottleneck
<a id="sec-build-bottleneck"></a>

Bottleneck ブロックの実装と shape 整合性を検証します。

In [20]:
# ===== 3-1-4: Bottleneck block implementation =====
from timm.layers import DropPath


class Bottleneck(nn.Module):
    """1x1x1 -> 3x3x3 -> 1x1x1 bottleneck residual block."""

    def __init__(self, inplanes, planes, stride=1, downsample=False, expansion_factor=4, drop_path_rate=0.0):
        super().__init__()
        out_channels = planes * expansion_factor

        # 1x1x1 reduction
        self.conv1 = nn.Conv3d(inplanes, planes, kernel_size=1, stride=1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)

        # 3x3x3 spatial conv (stride applied here)
        self.conv2 = nn.Conv3d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)

        # 1x1x1 expansion
        self.conv3 = nn.Conv3d(planes, out_channels, kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm3d(out_channels)

        self.relu = nn.ReLU(inplace=True)
        self.drop_path = DropPath(drop_prob=drop_path_rate) if drop_path_rate > 0.0 else nn.Identity()

        if isinstance(downsample, nn.Module):
            self.downsample = downsample
        elif downsample:
            self.downsample = nn.Sequential(
                nn.Conv3d(inplanes, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels),
            )
        else:
            self.downsample = None

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)
        out = self.drop_path(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out = out + residual
        out = self.relu(out)
        return out


# 3-1-4 validation A: no downsample (inplanes == planes * expansion_factor, stride=1)
x_bn = torch.randn(2, 64, 8, 32, 32)
bottleneck_no_ds = Bottleneck(
    inplanes=64,
    planes=16,  # out = 16 * expansion_factor = 64
    stride=1,
    downsample=False,
    expansion_factor=4,
    drop_path_rate=0.1,
).eval()
y_bn = bottleneck_no_ds(x_bn)
print(f"[bottleneck no_ds] input:  {tuple(x_bn.shape)}")
print(f"[bottleneck no_ds] output: {tuple(y_bn.shape)}")
assert y_bn.shape == x_bn.shape, "downsample=False では shape が一致する必要があります"

# 3-1-4 validation B: downsample with stride/channel change
x_bn_ds = torch.randn(2, 64, 8, 32, 32)
bottleneck_ds = Bottleneck(
    inplanes=64,
    planes=32,  # out = 32 * expansion_factor = 128
    stride=2,
    downsample=True,
    expansion_factor=4,
    drop_path_rate=0.1,
).eval()
y_bn_ds = bottleneck_ds(x_bn_ds)
print(f"[bottleneck ds] input:  {tuple(x_bn_ds.shape)}")
print(f"[bottleneck ds] output: {tuple(y_bn_ds.shape)}")
assert y_bn_ds.shape == (2, 128, 4, 16, 16), "expansion後チャネルとresidual shape を合わせる必要があります"

[bottleneck no_ds] input:  (2, 64, 8, 32, 32)
[bottleneck no_ds] output: (2, 64, 8, 32, 32)
[bottleneck ds] input:  (2, 64, 8, 32, 32)
[bottleneck ds] output: (2, 128, 4, 16, 16)


### 5-3. Encoder
<a id="sec-build-encoder"></a>

ResNet3D Encoder 本体（backbone 設定、DropPath、pretrained、checkpoint、forward_features、channels）を実装します。

In [21]:
# ===== 3-1-6/3-1-7/3-1-8/3-1-9/3-1-10/3-1-11/3-1-12/3-1-13/3-1-15: backbone + DropPath + pretrained + input channels + checkpoint + features + channels =====
from types import SimpleNamespace
from timm.models._manipulate import checkpoint


def conv_out_dim(in_size, kernel_size, stride=1, padding=0, dilation=1):
    return ((in_size + 2 * padding - dilation * (kernel_size - 1) - 1) // stride) + 1


def build_linear_drop_path_rates(total_blocks, max_rate):
    if total_blocks <= 0:
        return []
    if total_blocks == 1:
        return [float(max_rate)]
    return [float(max_rate) * i / (total_blocks - 1) for i in range(total_blocks)]


def resolve_pretrained_path(backbone_name: str):
    return f"./data/model_zoo/{backbone_name}_KM_200ep.pt"


def load_weights_stub(model, wpath):
    print(f"[load_weights] called: {wpath}")


class ResnetEncoder3d(nn.Module):
    """ResNet3D encoder with DropPath schedule, optional pretrained loading, input-channel conversion, gradient checkpointing, and feature/channel outputs."""

    BACKBONE_TABLE = {
        "r3d18": {"layers": [2, 2, 2, 2], "block": BasicBlock},
        "r3d200": {"layers": [3, 24, 36, 3], "block": Bottleneck},
    }

    def __init__(
        self,
        cfg,
        in_stride=(2, 2, 2),
        in_dilation=(1, 1, 1),
        drop_path_rate=0.2,
        inference_mode=False,
        load_weights_fn=load_weights_stub,
        use_checkpoint=False,
    ):
        super().__init__()
        self.cfg = cfg
        self.in_stride = in_stride
        self.in_dilation = in_dilation
        self.drop_path_rate = float(drop_path_rate)
        self.inference_mode = bool(inference_mode)
        self.load_weights_fn = load_weights_fn
        self.use_checkpoint = bool(use_checkpoint)
        self.pretrained_wpath = None

        backbone_name = cfg.backbone
        if backbone_name not in self.BACKBONE_TABLE:
            supported = ", ".join(self.BACKBONE_TABLE.keys())
            raise ValueError(f"ResnetEncoder3d backbone: {backbone_name} not implemented. supported=[{supported}]")

        spec = self.BACKBONE_TABLE[backbone_name]
        self.layers = spec["layers"]
        self.block = spec["block"]

        total_blocks = sum(self.layers)
        flat_rates = build_linear_drop_path_rates(total_blocks, self.drop_path_rate)
        self.block_drop_path_rates = []
        start = 0
        for n in self.layers:
            end = start + n
            self.block_drop_path_rates.append(flat_rates[start:end])
            start = end

        in_padding = tuple(d * 3 for d in in_dilation)
        # build as 3ch first, then convert to cfg.in_chans after optional weight load
        self.conv1 = nn.Conv3d(
            in_channels=3,
            out_channels=64,
            kernel_size=(7, 7, 7),
            stride=in_stride,
            dilation=in_dilation,
            padding=in_padding,
            bias=False,
        )
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=(3, 3, 3), stride=2, padding=1)

        self.inplanes = 64
        self.layer1 = self._make_layer(self.block, planes=64, n_blocks=self.layers[0], stride=1, block_drop_path_rates=self.block_drop_path_rates[0])
        self.layer2 = self._make_layer(self.block, planes=128, n_blocks=self.layers[1], stride=2, block_drop_path_rates=self.block_drop_path_rates[1])
        self.layer3 = self._make_layer(self.block, planes=256, n_blocks=self.layers[2], stride=2, block_drop_path_rates=self.block_drop_path_rates[2])
        self.layer4 = self._make_layer(self.block, planes=512, n_blocks=self.layers[3], stride=2, block_drop_path_rates=self.block_drop_path_rates[3])

        self._maybe_load_pretrained()
        self._update_input_channels()
        self.channels = self._infer_feature_channels()

    def _maybe_load_pretrained(self):
        self.pretrained_wpath = resolve_pretrained_path(self.cfg.backbone)
        if self.inference_mode:
            print("[pretrained] skipped because inference_mode=True")
            return
        self.load_weights_fn(self, self.pretrained_wpath)

    def _update_input_channels(self):
        """3-1-11: adapt stem conv1 to cfg.in_chans by mean+repeat."""
        target_ic = int(self.cfg.in_chans)
        if self.conv1.in_channels == target_ic:
            return
        old_conv = self.conv1
        with torch.no_grad():
            # average source input channels, then repeat to target channels
            w_mean = old_conv.weight.mean(dim=1, keepdim=True)
            w_new = w_mean.repeat(1, target_ic, 1, 1, 1)
        new_conv = nn.Conv3d(
            in_channels=target_ic,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            dilation=old_conv.dilation,
            bias=False,
        )
        new_conv.weight = nn.Parameter(w_new)
        self.conv1 = new_conv

    def _infer_feature_channels(self):
        """3-1-15: run forward_features with dummy input and cache channel dims."""
        with torch.no_grad():
            was_training = self.training
            self.eval()
            x_dummy = torch.randn(
                1,
                int(self.cfg.in_chans),
                32,
                96,
                96,
                device=self.conv1.weight.device,
                dtype=self.conv1.weight.dtype,
            )
            feats = self.forward_features(x_dummy)
            channels = [f.shape[1] for f in feats]
            self.train(was_training)
        return channels

    def _checkpoint_if_enabled(self, module, x):
        # 3-1-12: checkpoint is applied only in training mode
        if self.use_checkpoint and self.training:
            return checkpoint(module, x)
        return module(x)

    def _block_expansion(self):
        return 4 if self.block is Bottleneck else 1

    def _make_block(self, block, inplanes, planes, stride, downsample, drop_path_rate):
        if block is Bottleneck:
            return block(
                inplanes=inplanes,
                planes=planes,
                stride=stride,
                downsample=downsample,
                expansion_factor=4,
                drop_path_rate=drop_path_rate,
            )
        return block(
            inplanes=inplanes,
            planes=planes,
            stride=stride,
            downsample=downsample,
            drop_path_rate=drop_path_rate,
        )

    def _make_layer(self, block, planes, n_blocks, stride=1, block_drop_path_rates=None):
        if block_drop_path_rates is None:
            block_drop_path_rates = [0.0] * n_blocks
        if len(block_drop_path_rates) != n_blocks:
            raise ValueError(
                f"drop path rates length mismatch: got={len(block_drop_path_rates)} expected={n_blocks}"
            )
        expansion = self._block_expansion()
        out_channels = planes * expansion

        need_downsample = (stride != 1) or (self.inplanes != out_channels)
        downsample = None
        if need_downsample:
            downsample = nn.Sequential(
                nn.Conv3d(self.inplanes, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels),
            )

        layers = [
            self._make_block(
                block=block,
                inplanes=self.inplanes,
                planes=planes,
                stride=stride,
                downsample=downsample,
                drop_path_rate=block_drop_path_rates[0],
            )
        ]
        self.inplanes = out_channels
        for i in range(1, n_blocks):
            layers.append(
                self._make_block(
                    block=block,
                    inplanes=self.inplanes,
                    planes=planes,
                    stride=1,
                    downsample=None,
                    drop_path_rate=block_drop_path_rates[i],
                )
            )
        return nn.Sequential(*layers)

    def forward_features(self, x):
        """3-1-13: return [stem, layer1, layer2, layer3, layer4] features."""
        feats = []
        x = self._checkpoint_if_enabled(self.conv1, x)
        x = self.bn1(x)
        x = self.relu(x)
        feats.append(x)
        x = self.maxpool(x)
        x = self._checkpoint_if_enabled(self.layer1, x)
        feats.append(x)
        x = self._checkpoint_if_enabled(self.layer2, x)
        feats.append(x)
        x = self._checkpoint_if_enabled(self.layer3, x)
        feats.append(x)
        x = self._checkpoint_if_enabled(self.layer4, x)
        feats.append(x)
        return feats

    def forward(self, x):
        return self.forward_features(x)[-1]


def resolve_backbone(cfg):
    model = ResnetEncoder3d(cfg, inference_mode=True)
    return model.layers, model.block.__name__


def gather_block_drop_probs(model):
    probs = []
    for stage in [model.layer1, model.layer2, model.layer3, model.layer4]:
        for block in stage:
            dp = getattr(block, "drop_path", None)
            probs.append(float(getattr(dp, "drop_prob", 0.0)) if dp is not None else 0.0)
    return probs


class LoadRecorder:
    def __init__(self):
        self.calls = []

    def __call__(self, model, wpath):
        self.calls.append(wpath)


# validation A: backbone table
cfg18 = SimpleNamespace(backbone="r3d18", in_chans=1)
layers18, block18 = resolve_backbone(cfg18)
assert layers18 == [2, 2, 2, 2] and block18 == "BasicBlock"
cfg200 = SimpleNamespace(backbone="r3d200", in_chans=1)
layers200, block200 = resolve_backbone(cfg200)
assert layers200 == [3, 24, 36, 3] and block200 == "Bottleneck"

# validation B: unsupported backbone
try:
    _ = ResnetEncoder3d(SimpleNamespace(backbone="r3d999", in_chans=1), inference_mode=True)
    raise AssertionError("unsupported backbone で ValueError が必要です")
except ValueError:
    pass

# validation C: DropPath allocation
encoder18 = ResnetEncoder3d(cfg18, drop_path_rate=0.2, inference_mode=True).eval()
all_rates_18 = gather_block_drop_probs(encoder18)
assert len(all_rates_18) == sum(encoder18.layers)
assert all(all_rates_18[i] <= all_rates_18[i + 1] for i in range(len(all_rates_18) - 1))

# validation D: pretrained load behavior
rec_train = LoadRecorder()
_ = ResnetEncoder3d(cfg18, inference_mode=False, load_weights_fn=rec_train)
assert rec_train.calls == [resolve_pretrained_path("r3d18")]
rec_infer = LoadRecorder()
_ = ResnetEncoder3d(cfg18, inference_mode=True, load_weights_fn=rec_infer)
assert rec_infer.calls == []

# validation E (3-1-11): in_chans != 3 works and conv1.in_channels matches cfg
cfg1 = SimpleNamespace(backbone="r3d18", in_chans=1)
m1 = ResnetEncoder3d(cfg1, inference_mode=True).eval()
assert m1.conv1.in_channels == 1
with torch.no_grad():
    y1 = m1(torch.randn(1, 1, 32, 96, 96))
print(f"[3-1-11] in_chans=1 conv1.in_channels={m1.conv1.in_channels}, out={tuple(y1.shape)}")

cfg5 = SimpleNamespace(backbone="r3d18", in_chans=5)
m5 = ResnetEncoder3d(cfg5, inference_mode=True).eval()
assert m5.conv1.in_channels == 5
with torch.no_grad():
    y5 = m5(torch.randn(1, 5, 32, 96, 96))
print(f"[3-1-11] in_chans=5 conv1.in_channels={m5.conv1.in_channels}, out={tuple(y5.shape)}")

# validation F (3-1-12): use_checkpoint=True/False both run backward
cfg_ckpt = SimpleNamespace(backbone="r3d18", in_chans=1)
m_no_ckpt = ResnetEncoder3d(cfg_ckpt, inference_mode=True, use_checkpoint=False).train()
x_no_ckpt = torch.randn(1, 1, 16, 64, 64, requires_grad=True)
loss_no_ckpt = m_no_ckpt(x_no_ckpt).mean()
loss_no_ckpt.backward()
assert m_no_ckpt.conv1.weight.grad is not None

m_ckpt = ResnetEncoder3d(cfg_ckpt, inference_mode=True, use_checkpoint=True).train()
x_ckpt = torch.randn(1, 1, 16, 64, 64, requires_grad=True)
loss_ckpt = m_ckpt(x_ckpt).mean()
loss_ckpt.backward()
assert m_ckpt.conv1.weight.grad is not None
print("[3-1-12] checkpoint off/on: backward passed")

# validation G (3-1-13): forward_features returns 5-stage outputs
cfg_feat = SimpleNamespace(backbone="r3d18", in_chans=1)
m_feat = ResnetEncoder3d(cfg_feat, inference_mode=True).eval()
with torch.no_grad():
    feats = m_feat.forward_features(torch.randn(1, 1, 32, 96, 96))
assert len(feats) == 5
feat_channels = [f.shape[1] for f in feats]
print(f"[3-1-13] n_feats={len(feats)}, channels={feat_channels}")

# validation H (3-1-15): self.channels is computed from forward_features
cfg_ch = SimpleNamespace(backbone="r3d18", in_chans=1)
m_ch = ResnetEncoder3d(cfg_ch, inference_mode=True).eval()
assert hasattr(m_ch, "channels")
assert len(m_ch.channels) == 5
with torch.no_grad():
    feats_ch = m_ch.forward_features(torch.randn(1, 1, 32, 96, 96))
expected_channels = [f.shape[1] for f in feats_ch]
assert m_ch.channels == expected_channels
print(f"[3-1-15] channels={m_ch.channels}")

[pretrained] skipped because inference_mode=True
[pretrained] skipped because inference_mode=True
[pretrained] skipped because inference_mode=True
[pretrained] skipped because inference_mode=True
[pretrained] skipped because inference_mode=True
[3-1-11] in_chans=1 conv1.in_channels=1, out=(1, 512, 1, 3, 3)
[pretrained] skipped because inference_mode=True
[3-1-11] in_chans=5 conv1.in_channels=5, out=(1, 512, 1, 3, 3)
[pretrained] skipped because inference_mode=True
[pretrained] skipped because inference_mode=True
[3-1-12] checkpoint off/on: backward passed
[pretrained] skipped because inference_mode=True
[3-1-13] n_feats=5, channels=[64, 64, 128, 256, 512]
[pretrained] skipped because inference_mode=True
[3-1-15] channels=[64, 64, 128, 256, 512]


### 5-4. Decoder Common Block (3-2-1)
<a id="sec-build-decoder-common"></a>

3-2-1として、UNetデコーダで使う `Conv3d -> BatchNorm3d -> ReLU` の共通ブロック `ConvBnAct3d` を実装します。

In [23]:
# ===== 3-2-1: ConvBnAct3d implementation =====
class ConvBnAct3d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        padding=0,
        stride=1,
        norm_layer=nn.BatchNorm3d,
        act_layer=nn.ReLU,
    ):
        super().__init__()
        self.conv = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.norm = norm_layer(out_channels)
        self.act = act_layer(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        return x


# 3-2-1 validation: shape check
x_cba = torch.randn(2, 32, 8, 24, 24)
m_cba = ConvBnAct3d(
    in_channels=32,
    out_channels=64,
    kernel_size=3,
    padding=1,
    stride=2,
).eval()
with torch.no_grad():
    y_cba = m_cba(x_cba)

print(f"[3-2-1] input shape:  {tuple(x_cba.shape)}")
print(f"[3-2-1] output shape: {tuple(y_cba.shape)}")
assert y_cba.shape == (2, 64, 4, 12, 12)

[3-2-1] input shape:  (2, 32, 8, 24, 24)
[3-2-1] output shape: (2, 64, 4, 12, 12)


### 5-5. Decoder
<a id="sec-build-decoder"></a>

3-2-2 と 3-2-3 を統合し、upsample経路とskip接続を持つ `DecoderBlock3d` を実装します。

In [26]:
# ===== 3-2-2 + 3-2-3: DecoderBlock3d (upsample + skip) implementation =====
from monai.networks.blocks import UpSample


class DecoderBlock3d(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        norm_layer=nn.BatchNorm3d,
        upsample_mode="deconv",
        scale_factor=2,
    ):
        super().__init__()
        self.skip_channels = int(skip_channels)
        self.upsample = UpSample(
            spatial_dims=3,
            in_channels=in_channels,
            out_channels=in_channels,
            scale_factor=scale_factor,
            mode=upsample_mode,
        )
        self.conv1 = ConvBnAct3d(
            in_channels=in_channels + self.skip_channels,
            out_channels=out_channels,
            kernel_size=3,
            padding=1,
            norm_layer=norm_layer,
        )
        self.conv2 = ConvBnAct3d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=3,
            padding=1,
            norm_layer=norm_layer,
        )

    def forward(self, x, skip=None):
        x = self.upsample(x)

        if skip is None and self.skip_channels > 0:
            skip = x.new_zeros((x.size(0), self.skip_channels, *x.shape[2:]))

        if skip is not None:
            if skip.shape[2:] != x.shape[2:]:
                raise ValueError(f"skip spatial mismatch: skip={tuple(skip.shape)} x={tuple(x.shape)}")
            x = torch.cat([x, skip], dim=1)

        x = self.conv1(x)
        x = self.conv2(x)
        return x


# validation A: with skip
x_main = torch.randn(2, 64, 4, 12, 12)
skip = torch.randn(2, 16, 8, 24, 24)
m_skip = DecoderBlock3d(
    in_channels=64,
    skip_channels=16,
    out_channels=32,
    upsample_mode="deconv",
    scale_factor=2,
).eval()
with torch.no_grad():
    y_skip = m_skip(x_main, skip=skip)
print(f"[3-2-3/with-skip] x={tuple(x_main.shape)}, skip={tuple(skip.shape)}, out={tuple(y_skip.shape)}")
assert y_skip.shape == (2, 32, 8, 24, 24)

# validation B: no skip (skip_channels=0)
m_no_skip = DecoderBlock3d(
    in_channels=64,
    skip_channels=0,
    out_channels=32,
    upsample_mode="nontrainable",
    scale_factor=2,
).eval()
with torch.no_grad():
    y_no_skip = m_no_skip(x_main, skip=None)
print(f"[3-2-3/no-skip] x={tuple(x_main.shape)}, out={tuple(y_no_skip.shape)}")
assert y_no_skip.shape == (2, 32, 8, 24, 24)

[3-2-3/with-skip] x=(2, 64, 4, 12, 12), skip=(2, 16, 8, 24, 24), out=(2, 32, 8, 24, 24)
[3-2-3/no-skip] x=(2, 64, 4, 12, 12), out=(2, 32, 8, 24, 24)


In [33]:
# ===== 3-2-4 + 3-2-5 + 3-2-6: channel design + block assembly + forward =====
def build_decoder_channel_plan(
    encoder_channels,
    decoder_channels=(256, 128, 64, 32, 16),
    skip_channels=None,
    scale_factors=(2, 2, 2, 2, 2),
):
    encoder_channels = list(encoder_channels)
    decoder_channels = list(decoder_channels)
    scale_factors = list(scale_factors)

    if len(encoder_channels) == 0:
        raise ValueError("encoder_channels must not be empty")

    n_stages = len(decoder_channels)
    if n_stages == 0:
        raise ValueError("decoder_channels must not be empty")

    if skip_channels is None:
        skip_channels = list(encoder_channels[1:]) + [0]
    else:
        skip_channels = list(skip_channels)

    if len(skip_channels) != n_stages:
        raise ValueError(
            f"skip_channels length mismatch: got={len(skip_channels)} expected={n_stages}"
        )
    if len(scale_factors) != n_stages:
        raise ValueError(
            f"scale_factors length mismatch: got={len(scale_factors)} expected={n_stages}"
        )

    in_channels = [encoder_channels[0]] + decoder_channels[:-1]
    stage_plan = []
    for idx, (ic, sc, dc, sf) in enumerate(
        zip(in_channels, skip_channels, decoder_channels, scale_factors),
        start=1,
    ):
        stage_plan.append(
            {
                "stage": idx,
                "in_channels": int(ic),
                "skip_channels": int(sc),
                "out_channels": int(dc),
                "scale_factor": int(sf),
            }
        )

    return stage_plan


class UnetDecoder3d(nn.Module):
    def __init__(
        self,
        encoder_channels,
        skip_channels=None,
        decoder_channels=(256, 128, 64, 32, 16),
        scale_factors=(2, 2, 2, 2, 2),
        norm_layer=nn.BatchNorm3d,
        upsample_mode="nontrainable",
    ):
        super().__init__()
        self.encoder_channels = tuple(encoder_channels)
        self.decoder_channels = tuple(decoder_channels)
        self.scale_factors = tuple(scale_factors)
        self.skip_channels = tuple(
            list(self.encoder_channels[1:]) + [0]
            if skip_channels is None
            else skip_channels
        )

        self.channel_plan = build_decoder_channel_plan(
            encoder_channels=self.encoder_channels,
            decoder_channels=self.decoder_channels,
            skip_channels=self.skip_channels,
            scale_factors=self.scale_factors,
        )

        self.in_channels = [s["in_channels"] for s in self.channel_plan]
        self.out_channels = [s["out_channels"] for s in self.channel_plan]

        # 3-2-5: build decoder blocks from stage-wise channel definitions
        self.blocks = nn.ModuleList(
            [
                DecoderBlock3d(
                    in_channels=stage["in_channels"],
                    skip_channels=stage["skip_channels"],
                    out_channels=stage["out_channels"],
                    norm_layer=norm_layer,
                    upsample_mode=upsample_mode,
                    scale_factor=stage["scale_factor"],
                )
                for stage in self.channel_plan
            ]
        )

    def forward(self, feats):
        # 3-2-6: decode multi-scale features and return stage-wise outputs
        if len(feats) == 0:
            raise ValueError("feats must contain at least one tensor")

        res = [feats[0]]
        skips = feats[1:]

        for i, block in enumerate(self.blocks):
            skip = skips[i] if i < len(skips) else None
            res.append(block(res[-1], skip=skip))

        return res


def extract_block_specs(decoder):
    specs = []
    for stage, block in zip(decoder.channel_plan, decoder.blocks):
        specs.append(
            {
                "conv1_in": int(block.conv1.conv.in_channels),
                "conv1_out": int(block.conv1.conv.out_channels),
                "skip_channels": int(block.skip_channels),
                "scale_factor": int(stage["scale_factor"]),
            }
        )
    return specs


# 3-2-4 validation A: default skip_channels behavior
plan = build_decoder_channel_plan(
    encoder_channels=[128, 64, 32, 16, 8],
    decoder_channels=(256, 128, 64, 32, 16),
    skip_channels=None,
    scale_factors=(2, 2, 2, 2, 2),
)
assert [s["in_channels"] for s in plan] == [128, 256, 128, 64, 32]
assert [s["skip_channels"] for s in plan] == [64, 32, 16, 8, 0]
assert [s["out_channels"] for s in plan] == [256, 128, 64, 32, 16]
print("[3-2-4/default]", plan)

# 3-2-4 validation B: length mismatch detection
try:
    _ = build_decoder_channel_plan(
        encoder_channels=[128, 64, 32, 16, 8],
        decoder_channels=(256, 128, 64),
        skip_channels=(64, 32),
        scale_factors=(2, 2, 2),
    )
    raise AssertionError("skip_channels mismatch should raise ValueError")
except ValueError:
    pass

# 3-2-5 validation A: number of blocks equals number of decoder stages
decoder = UnetDecoder3d(
    encoder_channels=[128, 64, 32, 16, 8],
    decoder_channels=(256, 128, 64, 32, 16),
    skip_channels=None,
    scale_factors=(2, 2, 2, 2, 2),
    upsample_mode="nontrainable",
)
assert len(decoder.blocks) == 5

# 3-2-5 validation B: per-stage channel settings are reflected
specs = extract_block_specs(decoder)
expected_conv1_in = [128 + 64, 256 + 32, 128 + 16, 64 + 8, 32 + 0]
expected_conv1_out = [256, 128, 64, 32, 16]
assert [s["conv1_in"] for s in specs] == expected_conv1_in
assert [s["conv1_out"] for s in specs] == expected_conv1_out

# 3-2-5 validation C: stage-wise scale_factor is reflected
assert [s["scale_factor"] for s in specs] == [2, 2, 2, 2, 2]
print("[3-2-5/blocks] n_blocks=", len(decoder.blocks))
print("[3-2-5/specs]", specs)

# 3-2-6 validation: forward returns stage outputs list in expected order
feats_dummy = [
    torch.randn(2, 128, 1, 4, 4),
    torch.randn(2, 64, 2, 8, 8),
    torch.randn(2, 32, 4, 16, 16),
    torch.randn(2, 16, 8, 32, 32),
    torch.randn(2, 8, 16, 64, 64),
]
with torch.no_grad():
    decoded = decoder(feats_dummy)

assert len(decoded) == 1 + len(decoder.blocks)
decoded_shapes = [tuple(x.shape) for x in decoded]
expected_shapes = [
    (2, 128, 1, 4, 4),
    (2, 256, 2, 8, 8),
    (2, 128, 4, 16, 16),
    (2, 64, 8, 32, 32),
    (2, 32, 16, 64, 64),
    (2, 16, 32, 128, 128),
]
assert decoded_shapes == expected_shapes
print("[3-2-6/decoded_shapes]", decoded_shapes)

[3-2-4/default] [{'stage': 1, 'in_channels': 128, 'skip_channels': 64, 'out_channels': 256, 'scale_factor': 2}, {'stage': 2, 'in_channels': 256, 'skip_channels': 32, 'out_channels': 128, 'scale_factor': 2}, {'stage': 3, 'in_channels': 128, 'skip_channels': 16, 'out_channels': 64, 'scale_factor': 2}, {'stage': 4, 'in_channels': 64, 'skip_channels': 8, 'out_channels': 32, 'scale_factor': 2}, {'stage': 5, 'in_channels': 32, 'skip_channels': 0, 'out_channels': 16, 'scale_factor': 2}]
[3-2-5/blocks] n_blocks= 5
[3-2-5/specs] [{'conv1_in': 192, 'conv1_out': 256, 'skip_channels': 64, 'scale_factor': 2}, {'conv1_in': 288, 'conv1_out': 128, 'skip_channels': 32, 'scale_factor': 2}, {'conv1_in': 144, 'conv1_out': 64, 'skip_channels': 16, 'scale_factor': 2}, {'conv1_in': 72, 'conv1_out': 32, 'skip_channels': 8, 'scale_factor': 2}, {'conv1_in': 32, 'conv1_out': 16, 'skip_channels': 0, 'scale_factor': 2}]
[3-2-6/decoded_shapes] [(2, 128, 1, 4, 4), (2, 256, 2, 8, 8), (2, 128, 4, 16, 16), (2, 64, 8, 3

In [34]:
# ===== 3-2-7: SegmentationHead3d implementation =====
from monai.networks.blocks import UpSample


class SegmentationHead3d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        scale_factor=(2, 2, 2),
        upsample_mode="nontrainable",
    ):
        super().__init__()
        self.conv = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            padding=1,
        )

        self.upsample = UpSample(
            spatial_dims=3,
            in_channels=out_channels,
            out_channels=out_channels,
            scale_factor=scale_factor,
            mode=upsample_mode,
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.upsample(x)
        return x


# 3-2-7 validation A: in_channels -> out_channels conversion + final upsample
head = SegmentationHead3d(
    in_channels=16,
    out_channels=1,
    scale_factor=(2, 2, 2),
    upsample_mode="nontrainable",
).eval()
x_head = torch.randn(2, 16, 32, 128, 128)
with torch.no_grad():
    logits = head(x_head)

assert logits.shape == (2, 1, 64, 256, 256)
print("[3-2-7/logits_shape]", tuple(logits.shape))

# 3-2-7 validation B: no resize when scale_factor=(1,1,1)
head_no_up = SegmentationHead3d(
    in_channels=16,
    out_channels=3,
    scale_factor=(1, 1, 1),
    upsample_mode="nontrainable",
).eval()
with torch.no_grad():
    logits_no_up = head_no_up(x_head)

assert logits_no_up.shape == (2, 3, 32, 128, 128)
print("[3-2-7/no_up_shape]", tuple(logits_no_up.shape))

[3-2-7/logits_shape] (2, 1, 64, 256, 256)
[3-2-7/no_up_shape] (2, 3, 32, 128, 128)


### 5-6. Net Constructor I/O (3-3-1)
<a id="sec-build-net-constructor"></a>

3-3-1として、`Net` クラスのコンストラクタI/O仕様（必須/任意引数、既定値、上書き優先順位）を実装します。

In [37]:
# ===== 3-3-1: Net constructor I/O specification =====
from types import SimpleNamespace


class Net(nn.Module):
    """UNet3D wrapper model: constructor I/O contract only (forward will be added in 3-3-5)."""

    def __init__(
        self,
        cfg,
        num_classes=None,
        in_chans=None,
        deep_supervision=None,
        inference_mode=False,
    ):
        super().__init__()
        self.cfg = cfg
        self.inference_mode = bool(inference_mode)

        if not hasattr(cfg, "backbone"):
            raise ValueError("cfg.backbone is required")

        # I/O resolution rules:
        # 1) explicit argument has highest priority
        # 2) fallback to cfg field
        # 3) fallback to safe default
        self.num_classes = int(num_classes if num_classes is not None else getattr(cfg, "seg_classes", 1))
        self.in_chans = int(in_chans if in_chans is not None else getattr(cfg, "in_chans", 1))
        self.deep_supervision = bool(
            deep_supervision if deep_supervision is not None else getattr(cfg, "deep_supervision", False)
        )

        self.decoder_channels = tuple(getattr(cfg, "decoder_channels", (256, 128, 64, 32, 16)))
        self.scale_factors = tuple(getattr(cfg, "scale_factors", (2, 2, 2, 2, 2)))
        self.skip_channels = getattr(cfg, "skip_channels", None)
        self.upsample_mode = str(getattr(cfg, "upsample_mode", "nontrainable"))

        self.head_scale_factor = tuple(getattr(cfg, "head_scale_factor", (2, 2, 2)))
        self.head_upsample_mode = str(getattr(cfg, "head_upsample_mode", "nontrainable"))

        # Keep a clear spec map for downstream tickets and train/infer code.
        self.io_spec = {
            "required": ["cfg.backbone"],
            "optional_with_defaults": {
                "num_classes": self.num_classes,
                "in_chans": self.in_chans,
                "deep_supervision": self.deep_supervision,
                "decoder_channels": self.decoder_channels,
                "scale_factors": self.scale_factors,
                "skip_channels": self.skip_channels,
                "upsample_mode": self.upsample_mode,
                "head_scale_factor": self.head_scale_factor,
                "head_upsample_mode": self.head_upsample_mode,
                "inference_mode": self.inference_mode,
            },
        }

        encoder_cfg = SimpleNamespace(backbone=cfg.backbone, in_chans=self.in_chans)
        self.encoder = ResnetEncoder3d(
            cfg=encoder_cfg,
            inference_mode=self.inference_mode,
        )
        encoder_channels = list(self.encoder.channels[::-1])

        self.decoder = UnetDecoder3d(
            encoder_channels=encoder_channels,
            decoder_channels=self.decoder_channels,
            skip_channels=self.skip_channels,
            scale_factors=self.scale_factors,
            upsample_mode=self.upsample_mode,
        )

        self.seg_head = SegmentationHead3d(
            in_channels=self.decoder.decoder_channels[-1],
            out_channels=self.num_classes,
            scale_factor=self.head_scale_factor,
            upsample_mode=self.head_upsample_mode,
        )

        if self.deep_supervision:
            self.aux_head = SegmentationHead3d(
                in_channels=encoder_channels[0],
                out_channels=self.num_classes,
                scale_factor=(1, 1, 1),
                upsample_mode="nontrainable",
            )

    def forward(self, batch):
        raise NotImplementedError("3-3-5でforward本体を実装します")


# 3-3-1 validation A: minimal cfg should be enough to instantiate Net
cfg_331_min = SimpleNamespace(backbone="r3d18")
net_331_min = Net(cfg_331_min)
assert net_331_min.num_classes == 1
assert net_331_min.in_chans == 1
assert net_331_min.deep_supervision is False
assert isinstance(net_331_min.decoder_channels, tuple) and len(net_331_min.decoder_channels) == 5

# 3-3-1 validation B: explicit args must override cfg values
cfg_331_custom = SimpleNamespace(
    backbone="r3d18",
    seg_classes=3,
    in_chans=2,
    deep_supervision=True,
    decoder_channels=(192, 128, 64, 32, 16),
    scale_factors=(1, 2, 2, 2, 2),
    upsample_mode="deconv",
    head_scale_factor=(1, 1, 1),
)
net_331_custom = Net(
    cfg_331_custom,
    num_classes=4,
    in_chans=1,
    deep_supervision=False,
)
assert net_331_custom.num_classes == 4
assert net_331_custom.in_chans == 1
assert net_331_custom.deep_supervision is False
assert net_331_custom.upsample_mode == "deconv"
assert tuple(net_331_custom.scale_factors) == (1, 2, 2, 2, 2)

# 3-3-1 validation C: required field check
try:
    _ = Net(SimpleNamespace())
    raise AssertionError("cfg.backbone missing should raise ValueError")
except ValueError:
    pass

print("[3-3-1/minimal]", {
    "num_classes": net_331_min.num_classes,
    "in_chans": net_331_min.in_chans,
    "deep_supervision": net_331_min.deep_supervision,
})
print("[3-3-1/override]", {
    "num_classes": net_331_custom.num_classes,
    "in_chans": net_331_custom.in_chans,
    "deep_supervision": net_331_custom.deep_supervision,
})
print("[3-3-1/io_spec_keys]", list(net_331_min.io_spec.keys()))

[load_weights] called: ./data/model_zoo/r3d18_KM_200ep.pt
[load_weights] called: ./data/model_zoo/r3d18_KM_200ep.pt
[3-3-1/minimal] {'num_classes': 1, 'in_chans': 1, 'deep_supervision': False}
[3-3-1/override] {'num_classes': 4, 'in_chans': 1, 'deep_supervision': False}
[3-3-1/io_spec_keys] ['required', 'optional_with_defaults']


## 6. 学習・評価ループ（2-5）
<a id="sec-train"></a>

train/valid を分けて loss と最小評価指標（precision, recall, fbeta）を比較します。

In [ ]:
class Tiny3DAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(16, 8, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(8, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y


def binary_metrics_from_logits_or_probs(pred, target, threshold=0.5, beta=2.0):
    pred_bin = (pred >= threshold).to(torch.int64)
    tgt_bin = (target >= 0.5).to(torch.int64)

    tp = ((pred_bin == 1) & (tgt_bin == 1)).sum().item()
    fp = ((pred_bin == 1) & (tgt_bin == 0)).sum().item()
    fn = ((pred_bin == 0) & (tgt_bin == 1)).sum().item()

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    beta2 = beta * beta
    fbeta = (1 + beta2) * precision * recall / max(1e-8, beta2 * precision + recall)

    return {
        'precision': precision,
        'recall': recall,
        'fbeta': fbeta,
    }


def run_one_epoch(model, loader, optimizer, criterion, device, train=True):
    if train:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    metric_sum = {'precision': 0.0, 'recall': 0.0, 'fbeta': 0.0}
    n_steps = 0

    for step, batch in enumerate(loader, start=1):
        x = batch['input'].to(device)
        target = batch['target'].to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            y = model(x)
            loss = criterion(y, target)
            if train:
                loss.backward()
                optimizer.step()

        metrics = binary_metrics_from_logits_or_probs(y.detach(), target.detach(), threshold=0.5, beta=2.0)
        running_loss += loss.item()
        metric_sum['precision'] += metrics['precision']
        metric_sum['recall'] += metrics['recall']
        metric_sum['fbeta'] += metrics['fbeta']
        n_steps += 1

        mode = 'train' if train else 'valid'
        print(
            f'[{mode}] step={step}/{len(loader)} loss={loss.item():.6f} '
            f'P={metrics["precision"]:.4f} R={metrics["recall"]:.4f} F2={metrics["fbeta"]:.4f}'
        )

    epoch_loss = running_loss / max(1, n_steps)
    epoch_metrics = {k: v / max(1, n_steps) for k, v in metric_sum.items()}
    return epoch_loss, epoch_metrics


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Tiny3DAutoEncoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

print(f'[INFO] device = {device}')
print(f'[INFO] train_steps_per_epoch = {len(train_loader)}')
print(f'[INFO] valid_steps_per_epoch = {len(val_loader)}')

history = []
for epoch in range(1, EPOCHS + 1):
    train_loss, train_m = run_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        train=True,
    )
    val_loss, val_m = run_one_epoch(
        model=model,
        loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        train=False,
    )

    current_lr = optimizer.param_groups[0]['lr']
    row = {
        'epoch': epoch,
        'lr': current_lr,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_precision': train_m['precision'],
        'val_precision': val_m['precision'],
        'train_recall': train_m['recall'],
        'val_recall': val_m['recall'],
        'train_fbeta': train_m['fbeta'],
        'val_fbeta': val_m['fbeta'],
    }
    history.append(row)

    print(
        f'[epoch] {epoch} lr={current_lr:.2e} '
        f'train_loss={train_loss:.6f} val_loss={val_loss:.6f} '
        f'train_F2={train_m["fbeta"]:.4f} val_F2={val_m["fbeta"]:.4f}'
    )

history_df = pd.DataFrame(history)
display(history_df)

[INFO] device = cpu
[INFO] train_steps_per_epoch = 3
[INFO] valid_steps_per_epoch = 1
[train] step=1/3 loss=0.716268 P=0.0000 R=0.0000 F2=0.0000
[train] step=2/3 loss=0.712193 P=0.0002 R=1.0000 F2=0.0009
[train] step=3/3 loss=0.707416 P=0.0002 R=1.0000 F2=0.0010
[valid] step=1/1 loss=0.701552 P=0.0007 R=0.8942 F2=0.0035
[epoch] 1 lr=1.00e-03 train_loss=0.711959 val_loss=0.701552 train_F2=0.0006 val_F2=0.0035
[train] step=1/3 loss=0.701815 P=0.0002 R=1.0000 F2=0.0010
[train] step=2/3 loss=0.695607 P=0.0002 R=0.7778 F2=0.0011
[train] step=3/3 loss=0.684308 P=0.0000 R=0.0000 F2=0.0000
[valid] step=1/1 loss=0.671082 P=0.0000 R=0.0000 F2=0.0000
[epoch] 2 lr=1.00e-03 train_loss=0.693910 val_loss=0.671082 train_F2=0.0007 val_F2=0.0000
[train] step=1/3 loss=0.671730 P=0.0001 R=0.0370 F2=0.0004
[train] step=2/3 loss=0.658511 P=0.0000 R=0.0000 F2=0.0000
[train] step=3/3 loss=0.630833 P=0.0000 R=0.0000 F2=0.0000
[valid] step=1/1 loss=0.600669 P=0.0000 R=0.0000 F2=0.0000
[epoch] 3 lr=1.00e-03 trai

,epoch,lr,train_loss,val_loss,train_precision,val_precision,train_recall,val_recall,train_fbeta,val_fbeta
0,1,0.001,0.711959,0.701552,0.000125,0.000706,0.666667,0.894231,0.000623,0.003518
1,2,0.001,0.693910,0.671082,0.000140,0.000000,0.592593,0.000000,0.000701,0.000000
2,3,0.001,0.653691,0.600669,0.000030,0.000000,0.012346,0.000000,0.000150,0.000000
